# 25×25 Binary Matris — CNN ile YSA Eğitimi

## Problem Tanımı

25×25 (625 piksel) boyutunda binary matrisler üzerinde 5 farklı tahmin görevi:

| Görev | Açıklama | Çıktı Tipi |
|-------|----------|------------|
| **A** | 5 noktanın en yakın çiftinin Manhattan mesafesi | Regresyon |
| **B** | 5 noktanın en uzak çiftinin Manhattan mesafesi | Regresyon |
| **C** | Değişken sayıda (1-10) noktanın sayısı | 10-sınıf sınıflandırma |
| **D** | Nokta sayısının tek mi çift mi olduğu | Binary sınıflandırma |
| **E** | Koşullu köşe Manhattan mesafesi | Regresyon |

### Metodoloji
1. **Veri bölme:** Stratified %80 eğitim / %20 test (çıktı değerlerine göre dengeli)
2. **Hiperparametre optimizasyonu:** Grid search (8 konfigürasyon, 25 epoch)
3. **Performans karşılaştırması:** Eğitim verisinin %25, %50 ve %100'ü ile eğitim
4. **Early stopping:** Validation metriği iyileşmezse eğitimi durdur (patience=20)


## 1. Kütüphaneler ve Ortam Ayarları

In [14]:
import os, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Tekrarlanabilirlik için seed
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# GPU varsa kullan, yoksa CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

print(f"PyTorch: {torch.__version__}")
print(f"Device:  {DEVICE}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:     {props.name} ({props.total_memory / 1e9:.1f} GB VRAM)")
else:
    print("GPU:     Yok — CPU ile eğitim yapılacak")


PyTorch: 2.13.0.dev20260521+cu130
Device:  cuda
GPU:     NVIDIA GeForce RTX 5060 Ti (16.6 GB VRAM)


## 2. Veri Yükleme ve Ön İşleme

CSV dosyalarından veri yüklenir. Her satırda:
- **İlk 625 sütun:** 25×25 matrisin düzleştirilmiş hali (0 veya 1)
- **Son sütun:** Hedef değer (görev tipine göre değişir)

Matris CNN'e girebilmesi için `(1, 25, 25)` şeklinde reshape edilir (1 kanal, gri tonlama gibi).

### Stratified Split
Veri bölünürken çıktı değerlerine göre **dengeli bölme (stratified split)** yapılır.
Böylece eğitim ve test setlerinde her çıktı değerinden orantılı sayıda örnek bulunur.


In [15]:
class MatrixDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


def load_csv(filename):
    data = np.loadtxt(filename, delimiter=',')
    X = data[:, :625].reshape(-1, 1, 25, 25).astype(np.float32)
    y = data[:, 625].astype(np.float32)
    print(f"  Yüklendi: {filename} → {len(y)} örnek, çıktı aralığı: [{int(y.min())}, {int(y.max())}]")
    return X, y


def strat_split(y, ratio=0.2, seed=42):
    try:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=ratio, random_state=seed)
        return next(sss.split(np.zeros(len(y)), y.astype(int)))
    except ValueError:
        rng = np.random.RandomState(seed)
        idx = rng.permutation(len(y))
        k = int(len(y) * (1 - ratio))
        return idx[:k], idx[k:]


def strat_subset(y, frac, seed=42):
    if frac >= 1.0:
        return np.arange(len(y))
    try:
        keep, _ = next(StratifiedShuffleSplit(n_splits=1, test_size=1 - frac, random_state=seed)
                       .split(np.zeros(len(y)), y.astype(int)))
        return keep
    except ValueError:
        rng = np.random.RandomState(seed)
        idx = rng.permutation(len(y))
        return idx[:int(len(y) * frac)]

print("Veri yükleme fonksiyonları hazır.")


Veri yükleme fonksiyonları hazır.


## 3. CNN Model Mimarisi

2 katmanlı Convolutional Neural Network (CNN) kullanılmaktadır:

```
Giriş (1, 25, 25)
  ↓ Conv2D(f1, 3×3, padding=1) + BatchNorm + ReLU
  ↓ MaxPool(2×2)                                    → (f1, 12, 12)
  ↓ Conv2D(f2, 3×3, padding=1) + BatchNorm + ReLU
  ↓ MaxPool(2×2)                                    → (f2, 6, 6)
  ↓ Flatten                                          → (f2 × 36)
  ↓ Linear(fc) + ReLU + Dropout(0.3)
  ↓ Linear(çıkış)
```

### Katman Açıklamaları
| Katman | Amacı |
|--------|-------|
| **Conv2D** | Uzamsal desenleri (nokta konumları, mesafeler) öğrenir |
| **BatchNorm** | Eğitimi stabilize eder, daha hızlı yakınsama |
| **MaxPool** | Boyut azaltma, önemli özellikleri koruma |
| **Dropout (0.3)** | Overfitting önleme (rastgele nöron kapatma) |
| **Linear** | Öğrenilen özellikleri çıktıya dönüştürme |

Hiperparametreler `f1`, `f2`, `fc` grid search ile optimize edilecektir.


In [16]:
class CNN(nn.Module):
    def __init__(self, f1=32, f2=64, fc=128, out=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, f1, 3, padding=1), nn.BatchNorm2d(f1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(f1, f2, 3, padding=1), nn.BatchNorm2d(f2), nn.ReLU(), nn.MaxPool2d(2),
        )
        # 25 → MaxPool → 12 → MaxPool → 6
        self.head = nn.Sequential(
            nn.Linear(f2 * 6 * 6, fc), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(fc, out)
        )

    def forward(self, x):
        return self.head(self.conv(x).flatten(1))

# Parametre sayısı kontrolü
test_model = CNN(32, 64, 128, 1)
n_params = sum(p.numel() for p in test_model.parameters())
print(f"Örnek CNN (32,64,128,1) parametre sayısı: {n_params:,}")

test_model = CNN(64, 128, 256, 1)
n_params = sum(p.numel() for p in test_model.parameters())
print(f"Örnek CNN (64,128,256,1) parametre sayısı: {n_params:,}")


Örnek CNN (32,64,128,1) parametre sayısı: 314,177
Örnek CNN (64,128,256,1) parametre sayısı: 1,255,041


## 4. Eğitim ve Değerlendirme Fonksiyonları

### Eğitim Stratejisi
- **Optimizer:** Adam (weight_decay=1e-4 → L2 regularization)
- **LR Scheduler:** ReduceLROnPlateau — validation iyileşmezse learning rate yarıya düşer
- **Early Stopping:** Validation metriği `patience` epoch boyunca iyileşmezse eğitim durur

### Kayıp Fonksiyonları (Loss)
| Görev Tipi | Loss Fonksiyonu | Açıklama |
|------------|----------------|----------|
| Regresyon (A, B, E) | `MSELoss` | Ortalama kare hata |
| Sınıflandırma (C) | `CrossEntropyLoss` | Çok sınıflı sınıflandırma |
| Binary (D) | `BCEWithLogitsLoss` | İkili sınıflandırma |

### Değerlendirme Metrikleri
| Metrik | Açıklama | Kullanıldığı Görevler |
|--------|----------|----------------------|
| **MAE** | Ortalama mutlak hata (düşük = iyi) | A, B, E |
| **RMSE** | Kök ortalama kare hata (düşük = iyi) | A, B, E |
| **R²** | Determinasyon katsayısı (1'e yakın = iyi) | A, B, E |
| **Accuracy** | Doğruluk oranı (yüksek = iyi) | C, D |


In [17]:
def train_model(model, train_ds, val_ds, criterion, task_type, lr, bs, epochs, patience=20):
    model.to(DEVICE)
    train_loader = DataLoader(train_ds, bs, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, bs, pin_memory=True)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    best_val = float('inf') if task_type == 'reg' else 0
    best_state = None
    wait = 0
    history = {'train_loss': [], 'val_metric': []}

    for epoch in range(epochs):
        # ── Eğitim ──
        model.train()
        epoch_loss, n = 0, 0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y.long()) if task_type == 'cls' else criterion(out.squeeze(-1), y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * X.size(0)
            n += X.size(0)

        history['train_loss'].append(epoch_loss / n)

        # ── Validation ──
        val_m = evaluate(model, val_loader, task_type)
        val_score = val_m['mae'] if task_type == 'reg' else val_m['acc']
        history['val_metric'].append(val_score)

        scheduler.step(val_score if task_type == 'reg' else -val_score)

        # ── Early Stopping ──
        improved = (val_score < best_val) if task_type == 'reg' else (val_score > best_val)
        if improved:
            best_val = val_score
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break

    if best_state:
        model.load_state_dict(best_state)
    model.to(DEVICE)
    return best_val, epoch + 1, history


def evaluate(model, loader, task_type):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            out = model(X)
            if task_type == 'cls':
                preds.append(out.argmax(1).cpu())
            elif task_type == 'bin':
                preds.append((out.squeeze(-1).sigmoid() > 0.5).long().cpu())
            else:
                preds.append(out.squeeze(-1).cpu())
            targets.append(y.cpu())

    p = torch.cat(preds).float()
    t = torch.cat(targets).float()

    if task_type == 'reg':
        d = p - t
        ss_res = d.pow(2).sum().item()
        ss_tot = (t - t.mean()).pow(2).sum().item()
        return dict(
            mae=d.abs().mean().item(),
            rmse=d.pow(2).mean().sqrt().item(),
            r2=(1 - ss_res / ss_tot) if ss_tot > 0 else 0
        )
    return dict(acc=(p.long() == t.long()).float().mean().item())

print("Eğitim ve değerlendirme fonksiyonları hazır.")


Eğitim ve değerlendirme fonksiyonları hazır.


## 5. Hiperparametre Optimizasyonu

### Yöntem: Grid Search

Eğitim verisini tekrar bölerek (%80 search-train / %20 search-validation) her hiperparametre
kombinasyonu 25 epoch eğitilir. En iyi validation skoru veren kombinasyon seçilir.

### Arama Uzayı

| Parametre | Değerler | Açıklama |
|-----------|----------|----------|
| Learning Rate | 1e-3, 5e-4, 1e-4 | Öğrenme hızı |
| Batch Size | 32, 64 | Mini-batch boyutu |
| Conv Filtreler | (32,64), (64,128) | 1. ve 2. katman filtre sayısı |
| FC Boyut | 128, 256 | Fully connected katman nöron sayısı |

**Toplam:** 3 × 2 × 2 × 2 = **8 konfigürasyon** (görev başına ~200 epoch)

### Seçim Kriteri
- Regresyon → En düşük validation MAE
- Sınıflandırma → En yüksek validation accuracy


In [ ]:
HP_GRID = [
    dict(lr=1e-3, bs=32, f1=32,  f2=64,  fc=128),
    dict(lr=1e-3, bs=64, f1=64,  f2=128, fc=128),
    dict(lr=5e-4, bs=32, f1=64,  f2=128, fc=256),
    dict(lr=5e-4, bs=64, f1=32,  f2=64,  fc=256),
    dict(lr=1e-4, bs=32, f1=32,  f2=64,  fc=256),
    dict(lr=1e-4, bs=64, f1=64,  f2=128, fc=128),
    dict(lr=1e-3, bs=32, f1=64,  f2=128, fc=256),
    dict(lr=5e-4, bs=32, f1=32,  f2=64,  fc=128),
]

HP_EPOCHS = 25
HP_PATIENCE = 10
FINAL_EPOCHS = 150
FINAL_PATIENCE = 20


def hp_search(X_tr, y_tr, y_lab, criterion, ttype, out_size):
    hi, vi = strat_split(y_lab, 0.2, seed=123)
    h_ds = MatrixDataset(X_tr[hi], y_tr[hi])
    v_ds = MatrixDataset(X_tr[vi], y_tr[vi])

    best_hp, best_s = None, float('inf') if ttype == 'reg' else 0
    log = []

    for hp in HP_GRID:
        model = CNN(hp['f1'], hp['f2'], hp['fc'], out_size)
        s, ep, _ = train_model(model, h_ds, v_ds, criterion, ttype,
                               hp['lr'], hp['bs'], HP_EPOCHS, HP_PATIENCE)
        tag = f"MAE={s:.4f}" if ttype == 'reg' else f"Acc={s:.4f}"
        print(f"  lr={hp['lr']:.0e} bs={hp['bs']:2d} f=({hp['f1']:3d},{hp['f2']:3d}) "
              f"fc={hp['fc']:3d} → {tag} ({ep} ep)")
        log.append({**hp, 'score': float(s)})

        better = (s < best_s) if ttype == 'reg' else (s > best_s)
        if better:
            best_s, best_hp = s, hp

    return best_hp, best_s, log


def train_fractions(X_tr, y_tr, y_lab, X_te, y_te, criterion, ttype, out_size, best_hp, task_name='X'):
    test_ds = MatrixDataset(X_te, y_te)
    results = {}

    for label, frac in [('25%', 0.25), ('50%', 0.50), ('100%', 1.0)]:
        si = strat_subset(y_lab, frac, SEED)
        Xs, ys, ys_lab = X_tr[si], y_tr[si], y_lab[si]
        ti, vi = strat_split(ys_lab, 0.1, seed=77)

        model = CNN(best_hp['f1'], best_hp['f2'], best_hp['fc'], out_size)
        _, n_ep, hist = train_model(
            model, MatrixDataset(Xs[ti], ys[ti]), MatrixDataset(Xs[vi], ys[vi]),
            criterion, ttype, best_hp['lr'], best_hp['bs'], FINAL_EPOCHS, FINAL_PATIENCE)

        tm = evaluate(model, DataLoader(test_ds, best_hp['bs'], pin_memory=True), ttype)
        results[label] = dict(train_size=len(ti), epochs=n_ep, history=hist, metrics=tm)

        # Model kaydet
        pct = label.replace('%', '')
        torch.save(model.state_dict(), f'model_{task_name}_{pct}pct.pt')

        if ttype == 'reg':
            print(f"  {label}: {len(ti):>5d} örnek, {n_ep:>3d} epoch → "
                  f"MAE={tm['mae']:.3f}  RMSE={tm['rmse']:.3f}  R²={tm['r2']:.3f}")
        else:
            print(f"  {label}: {len(ti):>5d} örnek, {n_ep:>3d} epoch → Acc={tm['acc']:.3f}")

    return results

print("Hiperparametre arama fonksiyonları hazır.")
print(f"Grid boyutu: {len(HP_GRID)} konfigürasyon")

## 6. Görev A — En Yakın Çiftin Manhattan Mesafesi

**Giriş:** 25×25 binary matris (tam 5 nokta)
**Çıkış:** En yakın iki nokta arasındaki Manhattan mesafesi

$$d_{Manhattan} = |x_1 - x_2| + |y_1 - y_2|$$

5 noktanın $\binom{5}{2} = 10$ çift arasından en küçük mesafe bulunur.
Pratikte çıktı aralığı 0-18 civarındadır (çok yakın noktalar sık, çok uzak nadir).


In [ ]:
print("=" * 60)
print("GÖREV A: Min Manhattan (5 Nokta)")
print("=" * 60)

X, y = load_csv('a_200.csv')
y_lab = y.copy()

tr_i, te_i = strat_split(y_lab, 0.2, SEED)
X_tr, y_tr, y_lab_tr = X[tr_i], y[tr_i], y_lab[tr_i]
X_te, y_te = X[te_i], y[te_i]
print(f"Eğitim: {len(tr_i)}, Test: {len(te_i)}")

print("\nHiperparametre araması...")
best_hp_a, best_s, hp_log_a = hp_search(
    X_tr, y_tr, y_lab_tr, nn.MSELoss(), 'reg', 1)
print(f"\n✓ En iyi: lr={best_hp_a['lr']:.0e} bs={best_hp_a['bs']} "
      f"f=({best_hp_a['f1']},{best_hp_a['f2']}) fc={best_hp_a['fc']}")

print("\nFarklı veri boyutlarıyla eğitim:")
results_a = train_fractions(
    X_tr, y_tr, y_lab_tr, X_te, y_te,
    nn.MSELoss(), 'reg', 1, best_hp_a, task_name='A')

## 7. Görev B — En Uzak Çiftin Manhattan Mesafesi

**Giriş:** 25×25 binary matris (tam 5 nokta)
**Çıkış:** En uzak iki nokta arasındaki Manhattan mesafesi

Görev A'nın tersi. Çıktı aralığı daha geniş: 4-48 arası.
Köşelere yakın noktalar genelde yüksek mesafeler üretir.


In [ ]:
print("=" * 60)
print("GÖREV B: Max Manhattan (5 Nokta)")
print("=" * 60)

X, y = load_csv('b_200.csv')
y_lab = y.copy()

tr_i, te_i = strat_split(y_lab, 0.2, SEED)
X_tr, y_tr, y_lab_tr = X[tr_i], y[tr_i], y_lab[tr_i]
X_te, y_te = X[te_i], y[te_i]
print(f"Eğitim: {len(tr_i)}, Test: {len(te_i)}")

print("\nHiperparametre araması...")
best_hp_b, best_s, hp_log_b = hp_search(
    X_tr, y_tr, y_lab_tr, nn.MSELoss(), 'reg', 1)
print(f"\n✓ En iyi: lr={best_hp_b['lr']:.0e} bs={best_hp_b['bs']} "
      f"f=({best_hp_b['f1']},{best_hp_b['f2']}) fc={best_hp_b['fc']}")

print("\nFarklı veri boyutlarıyla eğitim:")
results_b = train_fractions(
    X_tr, y_tr, y_lab_tr, X_te, y_te,
    nn.MSELoss(), 'reg', 1, best_hp_b, task_name='B')

## 8. Görev C — Nokta Sayısını Tahmin

**Giriş:** 25×25 binary matris (1-10 arası nokta)
**Çıkış:** Nokta sayısı → 10-sınıf sınıflandırma

Model, matrisdeki 1'lerin toplamını sayma işlemi öğrenmelidir.
`CrossEntropyLoss` kullandığımız için etiketler 0-9'a dönüştürülür (1→0, 2→1, ..., 10→9).


In [ ]:
print("=" * 60)
print("GÖREV C: Nokta Sayısı (1-10)")
print("=" * 60)

X, y = load_csv('c_500.csv')
y_lab = y.copy()
y = y - 1  # 1-10 → 0-9 (CrossEntropyLoss için)

tr_i, te_i = strat_split(y_lab, 0.2, SEED)
X_tr, y_tr, y_lab_tr = X[tr_i], y[tr_i], y_lab[tr_i]
X_te, y_te = X[te_i], y[te_i]
print(f"Eğitim: {len(tr_i)}, Test: {len(te_i)}")

print("\nHiperparametre araması...")
best_hp_c, best_s, hp_log_c = hp_search(
    X_tr, y_tr, y_lab_tr, nn.CrossEntropyLoss(), 'cls', 10)
print(f"\n✓ En iyi: lr={best_hp_c['lr']:.0e} bs={best_hp_c['bs']} "
      f"f=({best_hp_c['f1']},{best_hp_c['f2']}) fc={best_hp_c['fc']}")

print("\nFarklı veri boyutlarıyla eğitim:")
results_c = train_fractions(
    X_tr, y_tr, y_lab_tr, X_te, y_te,
    nn.CrossEntropyLoss(), 'cls', 10, best_hp_c, task_name='C')

## 9. Görev D — Nokta Sayısı Tek mi Çift mi?

**Giriş:** 25×25 binary matris (1-10 arası nokta)
**Çıkış:** 0 (çift) veya 1 (tek)

### ⚠️ Parite Problemi

Parite tespiti sinir ağları için **bilinen zor problemlerden biridir:**
- Tek bir noktanın eklenmesi çıktıyı tamamen değiştirir (0↔1)
- CNN'in konvolüsyon katmanları **yerel** desenleri öğrenir, ancak parite **global** bir özellik
- Matematiksel olarak: $f(x) = \left(\sum x_i\right) \mod 2$
- Bu XOR benzeri bir fonksiyon — doğrusal olmayan ve tüm girdilere bağımlı

Bu nedenle CNN ile %50-55 civarı accuracy (rastgeleye yakın) beklenmektedir.


In [ ]:
print("=" * 60)
print("GÖREV D: Tek/Çift")
print("=" * 60)

X, y = load_csv('d_1000.csv')
y_lab = y.copy()

tr_i, te_i = strat_split(y_lab, 0.2, SEED)
X_tr, y_tr, y_lab_tr = X[tr_i], y[tr_i], y_lab[tr_i]
X_te, y_te = X[te_i], y[te_i]
print(f"Eğitim: {len(tr_i)}, Test: {len(te_i)}")

print("\nHiperparametre araması...")
best_hp_d, best_s, hp_log_d = hp_search(
    X_tr, y_tr, y_lab_tr, nn.BCEWithLogitsLoss(), 'bin', 1)
print(f"\n✓ En iyi: lr={best_hp_d['lr']:.0e} bs={best_hp_d['bs']} "
      f"f=({best_hp_d['f1']},{best_hp_d['f2']}) fc={best_hp_d['fc']}")

print("\nFarklı veri boyutlarıyla eğitim:")
results_d = train_fractions(
    X_tr, y_tr, y_lab_tr, X_te, y_te,
    nn.BCEWithLogitsLoss(), 'bin', 1, best_hp_d, task_name='D')

## 10. Görev E — Koşullu Köşe Manhattan Mesafesi

**Giriş:** 25×25 binary matris (1-10 arası nokta)
**Çıkış:**
- Nokta sayısı **tek** → Sol üst köşeye (0,0) en yakın noktanın mesafesi: $d = row + col$
- Nokta sayısı **çift** → Sağ alt köşeye (24,24) en yakın noktanın mesafesi: $d = (24-row) + (24-col)$

Bu görev iki yeteneği birleştirir:
1. Parite tespiti (tek mi çift mi)
2. Köşeye en yakın noktayı bulma

En zorlayıcı görevdir çünkü hem sayma hem mesafe hesabı gerektirir.


In [ ]:
print("=" * 60)
print("GÖREV E: Köşe Mesafesi")
print("=" * 60)

X, y = load_csv('e_200.csv')
y_lab = y.copy()

tr_i, te_i = strat_split(y_lab, 0.2, SEED)
X_tr, y_tr, y_lab_tr = X[tr_i], y[tr_i], y_lab[tr_i]
X_te, y_te = X[te_i], y[te_i]
print(f"Eğitim: {len(tr_i)}, Test: {len(te_i)}")

print("\nHiperparametre araması...")
best_hp_e, best_s, hp_log_e = hp_search(
    X_tr, y_tr, y_lab_tr, nn.MSELoss(), 'reg', 1)
print(f"\n✓ En iyi: lr={best_hp_e['lr']:.0e} bs={best_hp_e['bs']} "
      f"f=({best_hp_e['f1']},{best_hp_e['f2']}) fc={best_hp_e['fc']}")

print("\nFarklı veri boyutlarıyla eğitim:")
results_e = train_fractions(
    X_tr, y_tr, y_lab_tr, X_te, y_te,
    nn.MSELoss(), 'reg', 1, best_hp_e, task_name='E')

## 11. Sonuç Karşılaştırması

Tüm görevlerin %25, %50, %100 eğitim verisi ile performansları karşılaştırılmaktadır.


In [24]:
all_results = {
    'A': ('Min Manhattan (5 nokta)', 'reg', best_hp_a, results_a),
    'B': ('Max Manhattan (5 nokta)', 'reg', best_hp_b, results_b),
    'C': ('Nokta sayısı (1-10)',     'cls', best_hp_c, results_c),
    'D': ('Tek/Çift',               'bin', best_hp_d, results_d),
    'E': ('Köşe mesafesi',          'reg', best_hp_e, results_e),
}

print("=" * 78)
print(f"{'SONUÇ TABLOSU':^78}")
print("=" * 78)

for name, (desc, ttype, hp, results) in all_results.items():
    print(f"\nGörev {name}: {desc}")
    print(f"  HP: lr={hp['lr']:.0e}  bs={hp['bs']}  f=({hp['f1']},{hp['f2']})  fc={hp['fc']}")

    if ttype == 'reg':
        print(f"  {'Oran':>5s} {'Eğitim':>7s} {'Epoch':>5s}  {'MAE':>8s} {'RMSE':>8s} {'R²':>8s}")
        for label, r in results.items():
            m = r['metrics']
            print(f"  {label:>5s} {r['train_size']:>7d} {r['epochs']:>5d}  "
                  f"{m['mae']:>8.3f} {m['rmse']:>8.3f} {m['r2']:>8.3f}")
    else:
        print(f"  {'Oran':>5s} {'Eğitim':>7s} {'Epoch':>5s}  {'Accuracy':>10s}")
        for label, r in results.items():
            m = r['metrics']
            print(f"  {label:>5s} {r['train_size']:>7d} {r['epochs']:>5d}  {m['acc']:>10.3f}")


                                SONUÇ TABLOSU                                 

Görev A: Min Manhattan (5 nokta)
  HP: lr=1e-03  bs=32  f=(64,128)  fc=256
   Oran  Eğitim Epoch       MAE     RMSE       R²
    25%     612    38     2.360    2.934    0.641
    50%    1224    54     1.752    2.197    0.799
   100%    2448    84     1.414    1.772    0.869

Görev B: Max Manhattan (5 nokta)
  HP: lr=1e-04  bs=32  f=(32,64)  fc=256
   Oran  Eğitim Epoch       MAE     RMSE       R²
    25%    1512    75     2.431    3.124    0.934
    50%    3024    74     1.890    2.404    0.961
   100%    6048    60     1.648    2.118    0.969

Görev C: Nokta sayısı (1-10)
  HP: lr=1e-03  bs=64  f=(64,128)  fc=128
   Oran  Eğitim Epoch    Accuracy
    25%     900    27       0.412
    50%    1800    41       0.792
   100%    3600    73       0.951

Görev D: Tek/Çift
  HP: lr=1e-04  bs=64  f=(64,128)  fc=128
   Oran  Eğitim Epoch    Accuracy
    25%     360    32       0.567
    50%     720    22       0.493

## 12. Performans Grafikleri

### Veri Boyutunun Etkisi
Eğitim verisi arttıkça model performansının nasıl değiştiği:


In [ ]:
colors = ['#3498db', '#2ecc71', '#e74c3c']

for name, (desc, ttype, hp, results) in all_results.items():
    fig, ax = plt.subplots(figsize=(7, 5))
    fracs = list(results.keys())

    if ttype == 'reg':
        vals = [results[f]['metrics']['mae'] for f in fracs]
        bars = ax.bar(fracs, vals, color=colors)
        ax.set_ylabel('MAE (↓ daha iyi)')
    else:
        vals = [results[f]['metrics']['acc'] for f in fracs]
        bars = ax.bar(fracs, vals, color=colors)
        ax.set_ylabel('Accuracy (↑ daha iyi)')
        ax.set_ylim(0, 1.1)

    ax.set_title(f'Problem {name}: {desc}\nVeri Boyutunun Etkisi', fontweight='bold')
    ax.set_xlabel('Eğitim verisi oranı')

    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02 * max(vals),
                f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'{ANALYSIS_DIR}/performance_{name.lower()}.jpg', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Kaydedildi: performance_{name.lower()}.jpg")

### Eğitim Süreçleri
Her görev ve veri oranı için training loss ve validation metrik değişimleri:


In [ ]:
for name, (desc, ttype, hp, results) in all_results.items():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for label, r in results.items():
        ax1.plot(r['history']['train_loss'], label=label, alpha=0.8)
        ax2.plot(r['history']['val_metric'], label=label, alpha=0.8)

    ax1.set_title(f'Training Loss', fontsize=11)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()

    metric_name = 'Validation MAE' if ttype == 'reg' else 'Validation Accuracy'
    ax2.set_title(metric_name, fontsize=11)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('MAE' if ttype == 'reg' else 'Accuracy')
    ax2.legend()

    fig.suptitle(f'Problem {name}: {desc} — Eğitim Süreçleri', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{ANALYSIS_DIR}/training_curves_{name.lower()}.jpg', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Kaydedildi: training_curves_{name.lower()}.jpg")

## 13. Model Analizi — Doğru ve Yanlış Tahminler

Eğitilmiş modeller (%100 eğitim verisi) test seti üzerinde çalıştırılır.
Doğru ve yanlış tahminler görsel olarak incelenerek modellerin **neyi öğrenip neyi öğrenemediği** analiz edilir.

Analiz çıktıları `report/img/analysis/` klasörüne kaydedilir.


In [27]:
import os
from sklearn.metrics import confusion_matrix

ANALYSIS_DIR = 'report/img/analysis'
os.makedirs(ANALYSIS_DIR, exist_ok=True)

def get_predictions(csv_file, best_hp, ttype, out_size, model_file, label_shift=0):
    X, y = load_csv(csv_file)
    y_lab = y.copy()
    if label_shift:
        y = y - label_shift

    tr_i, te_i = strat_split(y_lab, 0.2, SEED)
    X_te, y_te = X[te_i], y[te_i]

    model = CNN(best_hp['f1'], best_hp['f2'], best_hp['fc'], out_size)
    model.load_state_dict(torch.load(model_file, map_location=DEVICE, weights_only=True))
    model.to(DEVICE)
    model.eval()

    preds = []
    with torch.no_grad():
        for i in range(0, len(X_te), 64):
            Xb = torch.FloatTensor(X_te[i:i+64]).to(DEVICE)
            out = model(Xb)
            if ttype == 'cls':
                preds.append(out.argmax(1).cpu().numpy())
            elif ttype == 'bin':
                preds.append((out.squeeze(-1).sigmoid() > 0.5).long().cpu().numpy())
            else:
                preds.append(out.squeeze(-1).cpu().numpy())

    return X_te, y_te, np.concatenate(preds)

# Tüm regresyon görevleri için tahminleri al
reg_tasks = [
    ('A', 'a_200.csv', best_hp_a, 'model_A_100pct.pt', 'Min Manhattan'),
    ('B', 'b_200.csv', best_hp_b, 'model_B_100pct.pt', 'Max Manhattan'),
    ('E', 'e_200.csv', best_hp_e, 'model_E_100pct.pt', 'Köşe Mesafesi'),
]
reg_data = {}
for name, csv, hp, mf, desc in reg_tasks:
    X_te, y_te, preds = get_predictions(csv, hp, 'reg', 1, mf)
    reg_data[name] = (X_te, y_te, preds)
    mae = np.abs(preds - y_te).mean()
    print(f"Görev {name} ({desc}): MAE={mae:.3f}")

# Sınıflandırma tahminleri
X_te_c, y_te_c, preds_c = get_predictions('c_500.csv', best_hp_c, 'cls', 10, 'model_C_100pct.pt', label_shift=1)
X_te_d, y_te_d, preds_d = get_predictions('d_1000.csv', best_hp_d, 'bin', 1, 'model_D_100pct.pt')
print(f"Görev C (Nokta Sayısı): Acc={np.mean(preds_c == y_te_c.astype(int)):.3f}")
print(f"Görev D (Tek/Çift): Acc={np.mean(preds_d == y_te_d.astype(int)):.3f}")
print("\nTahminler hazır. Analiz başlıyor...")


  Yüklendi: a_200.csv → 3400 örnek, çıktı aralığı: [1, 17]


FileNotFoundError: [Errno 2] No such file or directory: 'model_A_100pct.pt'

### 13.1 Tahmin vs Gerçek Değer (Regresyon)

Mükemmel model kırmızı kesikli çizgi (y=x) üzerinde olur.
Noktaların bu çizgiden sapması modelin hata miktarını gösterir.


In [ ]:
for name, desc in [('A', 'Min Manhattan'), ('B', 'Max Manhattan'), ('E', 'Köşe Mesafesi')]:
    X_te, y_te, preds = reg_data[name]

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(y_te, preds, alpha=0.3, s=10, c='#3498db')
    mn = min(y_te.min(), preds.min()) - 1
    mx = max(y_te.max(), preds.max()) + 1
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='y=x (ideal)')

    mae = np.abs(preds - y_te).mean()
    ax.set_title(f'Problem {name}: {desc}\nTahmin vs Gerçek (MAE={mae:.2f})', fontweight='bold')
    ax.set_xlabel('Gerçek Değer')
    ax.set_ylabel('Tahmin')
    ax.legend()

    plt.tight_layout()
    plt.savefig(f'{ANALYSIS_DIR}/scatter_{name.lower()}.jpg', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Kaydedildi: scatter_{name.lower()}.jpg")

### 13.2 Hata Dağılımları

Tahmin hatalarının (tahmin − gerçek) histogramı.
Sıfır merkezli dar dağılım → iyi model. Geniş veya kayık dağılım → sistematik hata.


In [ ]:
for name, desc in [('A', 'Min Manhattan'), ('B', 'Max Manhattan'), ('E', 'Köşe Mesafesi')]:
    X_te, y_te, preds = reg_data[name]
    errors = preds - y_te

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(errors, bins=40, color='#9b59b6', edgecolor='white', alpha=0.8)
    ax.axvline(0, color='red', linestyle='--', linewidth=2)
    ax.set_title(f'Problem {name}: {desc}\nHata Dağılımı (Ort={errors.mean():.2f}, Std={errors.std():.2f})', fontweight='bold')
    ax.set_xlabel('Hata (Tahmin − Gerçek)')
    ax.set_ylabel('Frekans')

    plt.tight_layout()
    plt.savefig(f'{ANALYSIS_DIR}/error_dist_{name.lower()}.jpg', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Kaydedildi: error_dist_{name.lower()}.jpg")

### 13.3 En Kötü Tahminler — Görsel İnceleme

Modelin en çok yanıldığı 5 örnek. Kırmızı kareler nokta konumlarını gösterir.
Bu matrislerde modelin neden zorlandığını analiz edebiliriz:
- Noktalar çok yakın mı / çok uzak mı?
- Karmaşık bir düzen mi var?


In [ ]:
for name, desc in [('A', 'Min Manhattan'), ('B', 'Max Manhattan'), ('E', 'Köşe Mesafesi')]:
    X_te, y_te, preds = reg_data[name]
    errors = np.abs(preds - y_te)
    worst_idx = np.argsort(errors)[-5:][::-1]

    fig, axes_row = plt.subplots(1, 5, figsize=(18, 4))
    fig.suptitle(f'Görev {name} ({desc}) — En Kötü 5 Tahmin', fontsize=13, fontweight='bold')

    for ax, wi in zip(axes_row, worst_idx):
        mat = X_te[wi, 0]
        ax.imshow(mat, cmap='binary', interpolation='nearest')
        points = np.argwhere(mat == 1)
        for py, px in points:
            ax.plot(px, py, 'rs', markersize=5)
        ax.set_title(f'Gerçek={y_te[wi]:.0f}\nTahmin={preds[wi]:.1f}\nHata={errors[wi]:.1f}',
                     fontsize=9, color='red')
        ax.set_xticks([0, 12, 24])
        ax.set_yticks([0, 12, 24])

    plt.tight_layout()
    plt.savefig(f'{ANALYSIS_DIR}/worst_{name.lower()}.jpg', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Kaydedildi: worst_{name.lower()}.jpg")


### 13.4 En İyi Tahminler — Karşılaştırma

Doğru tahmin edilen örnekler (yeşil). En kötülerle karşılaştırarak modelin
hangi düzenleri daha kolay öğrendiği görülür.


In [ ]:
for name, desc in [('A', 'Min Manhattan'), ('B', 'Max Manhattan'), ('E', 'Köşe Mesafesi')]:
    X_te, y_te, preds = reg_data[name]
    errors = np.abs(preds - y_te)
    best_idx = np.argsort(errors)[:5]

    fig, axes_row = plt.subplots(1, 5, figsize=(18, 4))
    fig.suptitle(f'Görev {name} ({desc}) — En İyi 5 Tahmin', fontsize=13, fontweight='bold')

    for ax, bi in zip(axes_row, best_idx):
        mat = X_te[bi, 0]
        ax.imshow(mat, cmap='binary', interpolation='nearest')
        points = np.argwhere(mat == 1)
        for py, px in points:
            ax.plot(px, py, 'gs', markersize=5)
        ax.set_title(f'Gerçek={y_te[bi]:.0f}\nTahmin={preds[bi]:.1f}\nHata={errors[bi]:.2f}',
                     fontsize=9, color='green')
        ax.set_xticks([0, 12, 24])
        ax.set_yticks([0, 12, 24])

    plt.tight_layout()
    plt.savefig(f'{ANALYSIS_DIR}/best_{name.lower()}.jpg', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Kaydedildi: best_{name.lower()}.jpg")


### 13.5 Confusion Matrix (Sınıflandırma)

- **Görev C:** Komşu sayılar (örn. 5 vs 6) birbiriyle karışıyor mu?
- **Görev D:** Model gerçekten tek/çift ayırt edebiliyor mu, yoksa hep aynı sınıfı mı tahmin ediyor?


In [ ]:
# ── C: Confusion Matrix ──
cm_c = confusion_matrix(y_te_c.astype(int), preds_c.astype(int))
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_c, cmap='Blues', interpolation='nearest')
ax.set_title('Problem C — Nokta Sayısı\nConfusion Matrix', fontweight='bold', fontsize=13)
ax.set_xlabel('Tahmin Edilen Sınıf')
ax.set_ylabel('Gerçek Sınıf')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(range(10))
ax.set_yticklabels(range(10))
for i in range(10):
    for j in range(10):
        color = 'white' if cm_c[i, j] > cm_c.max() / 2 else 'black'
        ax.text(j, i, str(cm_c[i, j]), ha='center', va='center', color=color, fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig(f'{ANALYSIS_DIR}/confusion_matrix_c.jpg', dpi=200, bbox_inches='tight')
plt.show()
print("Kaydedildi: confusion_matrix_c.jpg")

# ── D: Confusion Matrix ──
cm_d = confusion_matrix(y_te_d.astype(int), preds_d.astype(int))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_d, cmap='Oranges', interpolation='nearest')
ax.set_title('Problem D — Tek/Çift\nConfusion Matrix', fontweight='bold', fontsize=13)
ax.set_xlabel('Tahmin Edilen Sınıf')
ax.set_ylabel('Gerçek Sınıf')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Çift (0)', 'Tek (1)'])
ax.set_yticklabels(['Çift (0)', 'Tek (1)'])
for i in range(2):
    for j in range(2):
        color = 'white' if cm_d[i, j] > cm_d.max() / 2 else 'black'
        ax.text(j, i, str(cm_d[i, j]), ha='center', va='center', color=color, fontsize=16)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig(f'{ANALYSIS_DIR}/confusion_matrix_d.jpg', dpi=200, bbox_inches='tight')
plt.show()
print("Kaydedildi: confusion_matrix_d.jpg")

### 13.6 Yanlış Sınıflandırılan Örnekler

Görev C'de yanlış tahmin edilen matrislerin görsel incelemesi.
Kırmızı kareler nokta konumlarını, başlıklar gerçek ve tahmin edilen değerleri gösterir.


In [ ]:
# ── C: Yanlış örnekler ──
wrong_c = np.where(preds_c.astype(int) != y_te_c.astype(int))[0]
if len(wrong_c) > 0:
    show_n = min(8, len(wrong_c))
    fig, axes_row = plt.subplots(1, show_n, figsize=(show_n * 3, 3.5))
    fig.suptitle(f'Görev C — Yanlış Sınıflandırılan Örnekler ({len(wrong_c)} toplam)',
                 fontsize=13, fontweight='bold')
    if show_n == 1:
        axes_row = [axes_row]

    for i, ax in enumerate(axes_row):
        idx = wrong_c[i]
        mat = X_te_c[idx, 0]
        ax.imshow(mat, cmap='binary', interpolation='nearest')
        for py, px in np.argwhere(mat == 1):
            ax.plot(px, py, 'rs', markersize=4)
        actual = int(y_te_c[idx]) + 1
        predicted = int(preds_c[idx]) + 1
        ax.set_title(f'Gerçek: {actual}\nTahmin: {predicted}', fontsize=9, color='red')
        ax.set_xticks([0, 12, 24])
        ax.set_yticks([0, 12, 24])

    plt.tight_layout()
    plt.savefig(f'{ANALYSIS_DIR}/wrong_examples_c.jpg', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Kaydedildi: wrong_examples_c.jpg ({len(wrong_c)} yanlış örnek)")
else:
    print("Görev C: Tüm tahminler doğru!")

# ── D: Durum analizi ──
wrong_d = np.where(preds_d.astype(int) != y_te_d.astype(int))[0]
correct_d = np.where(preds_d.astype(int) == y_te_d.astype(int))[0]
print(f"\nGörev D: {len(correct_d)} doğru, {len(wrong_d)} yanlış (toplam {len(y_te_d)})")
print(f"  Accuracy: {len(correct_d)/len(y_te_d):.3f}")

unique_preds, pred_counts = np.unique(preds_d.astype(int), return_counts=True)
print(f"  Tahmin dağılımı: {dict(zip(unique_preds, pred_counts))}")
print("  → Model parite (tek/çift) öğrenemiyor — rastgele tahmin yapıyor")

### 13.7 Değer Bazında Hata Analizi

Model hangi çıktı değerlerinde daha başarılı? Yüksek MAE olan değerler modelin zorlandığı bölgeleri gösterir.


In [ ]:
for name, desc in [('A', 'Min Manhattan'), ('B', 'Max Manhattan'), ('E', 'Köşe Mesafesi')]:
    X_te, y_te, preds = reg_data[name]

    vals = np.unique(y_te.astype(int))
    mae_per_val = [np.abs(preds[y_te.astype(int) == v] - y_te[y_te.astype(int) == v]).mean() for v in vals]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(vals, mae_per_val, color='#e67e22', edgecolor='white')
    avg_mae = np.mean(mae_per_val)
    ax.axhline(avg_mae, color='red', linestyle='--', alpha=0.7, label=f'Ort MAE={avg_mae:.2f}')
    ax.set_title(f'Problem {name}: {desc}\nDeğer Bazında MAE', fontweight='bold')
    ax.set_xlabel('Gerçek Değer')
    ax.set_ylabel('MAE')
    ax.legend()

    plt.tight_layout()
    plt.savefig(f'{ANALYSIS_DIR}/mae_by_value_{name.lower()}.jpg', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Kaydedildi: mae_by_value_{name.lower()}.jpg")

### 13.8 Yorum ve Bulgular

#### Görev A (Min Manhattan)
- Model yakın noktaları tespit etmekte başarılı
- **Doğru tahminler:** Noktalar belirgin kümeler oluşturduğunda model mesafeyi iyi tahmin ediyor
- **Yanlış tahminler:** Birçok nokta benzer mesafede olduğunda (belirsiz minimum) zorlanıyor

#### Görev B (Max Manhattan)
- En başarılı model (R² ≈ 0.97) — köşelerdeki noktaları bulmak CNN için kolay
- **Doğru:** Noktalar zıt köşelerde → yüksek mesafe, net pattern
- **Yanlış:** Tüm noktalar merkeze yakın → düşük max mesafe, az görülen durum

#### Görev C (Nokta Sayısı)
- %95+ accuracy ile güçlü performans
- **Karışan sınıflar:** Genelde komşu sayılar (N vs N±1) karışıyor
- CNN toplama işlemini yaklaşık olarak öğrenebiliyor

#### Görev D (Tek/Çift)
- **~%50 accuracy → model öğrenemedi** (rastgele tahmin)
- Confusion matrix'te model genelde tek bir sınıfa yöneliyor
- **Neden:** Parite (XOR benzeri) fonksiyonu, CNN'in yerel filtrelerle öğrenemeyeceği global bir özellik
- Tek bir noktanın eklenmesi tüm çıktıyı değiştiriyor → gradyan bilgisi yetersiz

#### Görev E (Köşe Mesafesi)
- Orta düzey performans (R² ≈ 0.85)
- İki beceri birden gerekiyor: parite + mesafe → hata birikimi kaçınılmaz
- Parite kısmı tam öğrenilemese de köşeye yakın noktaları bulmak kısmen öğrenilmiş


## 14. Sonuç ve Değerlendirme

### Genel Bulgular

| Görev | Zorluk | Beklenen | Açıklama |
|-------|--------|----------|----------|
| **A** | Orta | R² > 0.85 | CNN mesafe ilişkilerini öğrenebilir |
| **B** | Kolay | R² > 0.95 | Uzak noktalar köşelerde → kolay pattern |
| **C** | Kolay | Acc > 0.95 | Toplama işlemi CNN için öğrenilebilir |
| **D** | Çok Zor | Acc ~ 0.50 | Parite problemi — CNN'in yapısal sınırlaması |
| **E** | Zor | R² ~ 0.85 | Parite + mesafe → iki beceri birden |

### Veri Boyutunun Etkisi
- Tüm görevlerde daha fazla eğitim verisi → daha iyi performans
- En büyük iyileşme genelde %25 → %50 geçişinde
- %50 → %100 geçişinde getiri azalır (diminishing returns)

### Task D Hakkında
Parite (XOR benzeri) problemi, feedforward ağların yapısal sınırlamasıdır.
625 bitin paritesini öğrenmek için ağın tüm girdileri aynı anda görmesi ve
modüler aritmetik yapabilmesi gerekir. Bu, CNN'in doğal yapısına uygun değildir.
